# Sesión de nivelación — Python y pandas desde cero

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Educación Continua · lunes 19 de octubre

## Para quién es esta sesión
Para quien nunca ha programado, o vio Python hace tiempo y no se acuerda. Si usted ya programa, hoy es **repaso**: vaya a su ritmo, use los retos ⚡ y, si quiere, acompañe a alguien que va empezando. El miércoles volvemos al hilo del curso (Sesión 2, datos de Medellín) con todos en la misma página.

## La idea de hoy: dos sombreros
Vamos a hacer, en pequeño, el trabajo de dos oficios que en la práctica suelen ser la misma persona:

- 🔧 **Ingeniero/a de datos** (primera hora): consigue los datos, entiende qué hay en cada columna, limpia, transforma, junta tablas y **entrega una tabla confiable**. Para eso necesita las piezas básicas de Python (variables, listas, diccionarios, condiciones, repeticiones, funciones) y pandas.
- 🔬 **Científico/a de datos** (segunda hora): toma esa tabla, **hace preguntas**, las responde con gráficas que construye paso a paso y con números, y deja la tabla lista para que un modelo aprenda (lo que llamamos *preprocesamiento*: estandarizar, convertir categorías en números). Al final, un modelo de verdad.

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Ejecutar, leer y modificar celdas en Colab, y **leer un mensaje de error** sin asustarse.
2. Usar las piezas básicas de Python para describir datos: variables, listas, diccionarios, `if`, `for` y funciones.
3. Cargar una tabla con pandas y hacer la limpieza del ingeniero de datos: nombres, tipos, fechas, duplicados, vacíos, filtros, agrupaciones y uniones de tablas.
4. **Construir una gráfica** con la receta de cuatro pasos (datos → tipo → títulos → ajustes), elegir el tipo según la pregunta y guardarla para un informe.
5. Dejar una tabla lista para un modelo: *one-hot* para las categorías y estandarización para los números.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Cómo funciona este notebook
Está **completo**: no hay huecos obligatorios. Su trabajo es ejecutar las celdas **en orden** (`Shift + Enter`), leer lo que sale y, sobre todo, **predecir antes de ejecutar**. Los símbolos:

| Símbolo | Qué hacer |
|---|---|
| 🔍 **Predice** | escriba (o piense) su predicción *antes* de correr la celda siguiente |
| 🧪 **Juegue** | cambie un valor, quite un `#`, vuelva a correr. No se rompe nada |
| 📘 **Para entender** | la explicación de fondo; léala después de ver el ejemplo |
| ⚡ **Reto** | opcional, para quien va adelantado |

## Los datos de hoy
**32 833 canciones de Spotify** con sus características de audio (qué tan bailable, enérgica o "feliz" suena cada una), su género, su artista y su popularidad. Es un dataset de enero de 2020, armado por Kaylin Pavlik para *TidyTuesday* con la API pública de Spotify (vía el paquete `spotifyr`); por eso no hay canciones posteriores. ¿Por qué canciones y no los siniestros viales de Medellín? Porque hoy el protagonista es el **lenguaje**, no el problema, y porque todos tenemos una opinión sobre si el reggaetón es más bailable que el rock. El miércoles volvemos a Medellín.


## 0. Setup

Ejecute estas celdas tal cual. Cargan las librerías, consiguen los datos y definen las funciones que dibujan las figuras de hoy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Los datos y las figuras del curso viven en el repositorio del curso.
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVO = "spotify_canciones.csv"            # 32 833 canciones de Spotify (TidyTuesday 2020-01-21)
ARCHIVO_GENEROS = "spotify_generos.csv"      # tabla pequeña: código del género → nombre y descripción
ARCHIVOS_RETO = ["volcanes.csv", "erupciones.csv"]   # para el reto final (volcanes del mundo)

<details><summary>Celda de plomería (no hay que entenderla hoy): consigue los datos y las figuras del curso</summary>

Baja al disco de Colab (una sola vez por sesión) los CSV de hoy y `figuras.py`, el archivo del curso con las funciones que dibujan los algoritmos. Si el repositorio no responde, use el Plan B de abajo.
</details>

In [ ]:
def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo} del repositorio. Use el Plan B (celda siguiente).")
    return False

for nombre in [ARCHIVO, ARCHIVO_GENEROS] + ARCHIVOS_RETO:
    asegurar_datos(nombre, REPO + "datos/" + nombre)
asegurar_datos("figuras.py", REPO + "tools/figuras.py")
URL = ARCHIVO
try:
    import figuras as fg   # las figuras del curso: fg.pipeline(), fg.one_hot_ejemplo(), ...
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo: datos y figuras disponibles.")

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola si los datos llegaron; si no, le pedirá subir `spotify_canciones.csv` (y `spotify_generos.csv`) desde su computador (el instructor los comparte por el canal del curso) y a partir de ahí todo el notebook funciona igual.
</details>

In [ ]:
# Plan B: solo actúa si la descarga falló. Con "Ejecutar todo" no se detiene.
from pathlib import Path
if Path(URL).exists() and Path(ARCHIVO_GENEROS).exists():
    print("✓ Datos disponibles: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Falta algún archivo → suba los que compartió el instructor (spotify_canciones.csv y spotify_generos.csv).")
    subido = files.upload()
    if subido:
        print("Archivos subidos:", list(subido))

<details><summary>Plomería de las figuras de hoy (no hay que entenderla): esquemas de listas, diccionarios, tablas, <code>merge</code>, distancias y escalado</summary>

Son funciones de dibujo. Las de los algoritmos del curso viven en `figuras.py`; estas son solo para esta sesión.
</details>

In [ ]:
# Plomería: funciones que dibujan los esquemas de hoy. Solo ejecútela.
import textwrap
from matplotlib.patches import FancyBboxPatch

AZUL, ROJO, GRIS, VERDE, NARANJA, NEGRO = "#2E6FB7", "#C4463A", "#8C8C8C", "#2C8C5B", "#D97C1F", "#222222"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"]})


def _que_mirar(fig, texto):
    fig.text(0.5, -0.01, textwrap.fill("Qué mirar: " + texto, 115), ha="center", va="top",
             fontsize=10, style="italic", color=NEGRO)
    return fig


def _caja(ax, x, y, w, h, texto, color, tam=10, relleno=True, negrita=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=color if relleno else "white", ec=color, lw=1.6))
    ax.text(x + w / 2, y + h / 2, texto, ha="center", va="center", fontsize=tam,
            color="white" if relleno else NEGRO, fontweight="bold" if negrita else "normal")


def dos_sombreros():
    """Los dos oficios de hoy, como dos filas de pasos."""
    filas = [("INGENIERO/A DE DATOS", AZUL, ["conseguir", "entender", "limpiar", "transformar", "juntar", "entregar\nla tabla"]),
             ("CIENTÍFICO/A DE DATOS", VERDE, ["preguntar", "graficar", "resumir", "preparar\nX e y", "modelar", "contar la\nhistoria"])]
    fig, ax = plt.subplots(figsize=(12, 3.9)); ax.set_xlim(0, 12); ax.set_ylim(0, 3.9); ax.axis("off")
    for j, (titulo, color, pasos) in enumerate(filas):
        y = 2.75 - j * 2.1
        ax.text(0.1, y + 0.4, titulo, fontsize=11, fontweight="bold", color=color, va="center")
        for i, p in enumerate(pasos):
            x = 3.4 + i * 1.42
            _caja(ax, x, y, 1.22, 0.8, p, color, tam=9.5, relleno=(j == 0 and i == 5) or (j == 1 and i == 4))
            if i < len(pasos) - 1:
                ax.annotate("", xy=(x + 1.42, y + 0.4), xytext=(x + 1.22, y + 0.4),
                            arrowprops=dict(arrowstyle="->", color=GRIS, lw=1.4))
    # la entrega del ingeniero baja (en línea recta, por el pasillo entre las dos filas) hasta la primera caja del científico
    ax.annotate("", xy=(4.0, 1.5), xytext=(10.1, 2.7), arrowprops=dict(arrowstyle="->", color=NARANJA, lw=1.8, ls="--"))
    ax.text(7.6, 1.78, "la tabla confiable pasa de un sombrero al otro", fontsize=9.5, color=NARANJA, ha="center", style="italic")
    return _que_mirar(fig, "hoy recorremos las dos filas, en este orden. Las cajas rellenas son las que conectan con el resto "
                           "del curso: la tabla que entrega el ingeniero es la que modela el científico.")


def dibujar_lista(valores, nombre="lista"):
    """Una lista como casillas numeradas: índices desde el principio arriba, desde el final abajo."""
    n = len(valores)
    fig, ax = plt.subplots(figsize=(1.9 * n + 1.8, 2.3)); ax.set_xlim(0, 1.9 * n + 1.8); ax.set_ylim(0, 2.3); ax.axis("off")
    ax.text(0.1, 1.15, f"{nombre} =", fontsize=12, va="center", family="monospace")
    for i, v in enumerate(valores):
        x = 1.8 + i * 1.9
        _caja(ax, x, 0.75, 1.7, 0.8, str(v), AZUL, tam=10, relleno=False)
        ax.text(x + 0.85, 1.8, f"[{i}]", ha="center", fontsize=10, color=AZUL, fontweight="bold")
        ax.text(x + 0.85, 0.48, f"[{i - n}]", ha="center", fontsize=10, color=ROJO)
    ax.text(1.8, 2.12, "posición contando desde el principio (empieza en 0)", fontsize=8.5, color=AZUL)
    ax.text(1.8, 0.15, "posición contando desde el final (−1 es el último)", fontsize=8.5, color=ROJO)
    return _que_mirar(fig, f"la primera casilla es la [0], no la [1]; la última es la [{n - 1}] o, contando desde atrás, la [-1].")


def dibujar_diccionario(d, nombre="dic"):
    """Un diccionario como parejas llave → valor."""
    n = len(d)
    fig, ax = plt.subplots(figsize=(7.5, 0.62 * n + 1.1)); ax.set_xlim(0, 7.5); ax.set_ylim(0, 0.62 * n + 1.1); ax.axis("off")
    ax.text(0.1, 0.62 * n + 0.7, f"{nombre} = {{ ... }}", fontsize=12, va="center", family="monospace")
    for i, (k, v) in enumerate(d.items()):
        y = 0.62 * (n - 1 - i) + 0.3
        _caja(ax, 0.4, y, 2.4, 0.5, f'"{k}"', AZUL, tam=10, relleno=False)
        ax.annotate("", xy=(3.6, y + 0.25), xytext=(2.85, y + 0.25), arrowprops=dict(arrowstyle="->", color=GRIS, lw=1.4))
        _caja(ax, 3.7, y, 3.4, 0.5, repr(v), VERDE, tam=10, relleno=False)
    ax.text(1.6, 0.02, "llave", ha="center", fontsize=8.5, color=AZUL)
    ax.text(5.4, 0.02, "valor", ha="center", fontsize=8.5, color=VERDE)
    return _que_mirar(fig, 'a cada valor se llega por su llave, no por su posición: dic["artista"], no dic[1].')


def dibujar_tabla(registros):
    """Una lista de diccionarios ES una tabla: cada diccionario una fila, cada llave una columna."""
    tabla = pd.DataFrame(registros)
    n = len(registros)
    fig, axes = plt.subplots(1, 3, figsize=(14, 0.85 * n + 1.7), gridspec_kw={"width_ratios": [1.2, 0.12, 1]})
    ax = axes[0]; ax.axis("off"); ax.set_xlim(0, 7); ax.set_ylim(0, n * 1.2 + 0.3); ax.set_title("lista de diccionarios")
    for i, r in enumerate(registros):
        y = (n - 1 - i) * 1.2 + 0.2
        ax.add_patch(FancyBboxPatch((0.1, y), 6.8, 1.0, boxstyle="round,pad=0.02,rounding_size=0.08", fc="#F4F7FB", ec=AZUL, lw=1.2))
        partes = [f'"{k}": {repr(v)}' for k, v in r.items()]
        lineas = [", ".join(partes[:3]) + ("," if len(partes) > 3 else ""), ", ".join(partes[3:])]   # máximo 3 parejas por línea
        texto = "{" + ("\n ".join(l for l in lineas if l)) + "}"
        ax.text(0.3, y + 0.5, texto, fontsize=8, va="center", family="monospace")
    axes[1].axis("off"); axes[1].text(0.5, 0.5, "→", fontsize=30, ha="center", va="center", color=GRIS, transform=axes[1].transAxes)
    ax = axes[2]; ax.axis("off"); ax.set_title("DataFrame (tabla)")
    celdas = tabla.astype(object).where(tabla.notna(), "NaN").values
    t = ax.table(cellText=celdas, colLabels=list(tabla.columns), rowLabels=[str(i) for i in tabla.index], loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(9.5); t.scale(1, 1.7); t.auto_set_column_width(list(range(tabla.shape[1])))
    for (r, c), cell in t.get_celld().items():
        cell.set_edgecolor("#DDDDDD")
        if r == 0:
            cell.set_facecolor("#EAF1FA"); cell.set_text_props(fontweight="bold")
        elif cell.get_text().get_text() == "NaN":
            cell.set_facecolor("#FBE9E7")
    return _que_mirar(fig, "las llaves se vuelven columnas y cada diccionario, una fila. El número de la izquierda es el índice de la fila "
                           "(desde 0). Donde a un diccionario le faltaba una llave, la tabla queda con NaN: 'no hay dato'.")


def dibujar_merge(izq, der, llave):
    """merge: por cada fila de la izquierda, buscar su llave en la tabla de consulta y traer sus columnas."""
    res = izq.merge(der, on=llave, how="left")
    alto = 0.42 * max(len(izq), len(res)) + 1.8
    fig, axes = plt.subplots(1, 3, figsize=(14, alto), gridspec_kw={"width_ratios": [1.1, 1.2, 1.9]})
    titulos = ["tabla izquierda (muchas filas)", f"tabla de consulta (una fila por {llave})", f'resultado de .merge(..., on="{llave}")']
    for ax, tabla, titulo in zip(axes, [izq, der, res], titulos):
        ax.axis("off"); ax.set_title(titulo, fontsize=10)
        t = ax.table(cellText=tabla.values, colLabels=list(tabla.columns), loc="center", cellLoc="center")
        t.auto_set_font_size(False); t.set_fontsize(9.5); t.scale(1, 1.6); t.auto_set_column_width(list(range(tabla.shape[1])))
        for (r, c), cell in t.get_celld().items():
            cell.set_edgecolor("#DDDDDD")
            if r == 0:
                cell.set_facecolor("#EAF1FA"); cell.set_text_props(fontweight="bold")
            elif list(tabla.columns)[c] == llave:
                cell.set_facecolor("#FFF4CC")
    return _que_mirar(fig, f"la columna amarilla ({llave}) es la llave. Por cada fila de la izquierda, merge busca su {llave} en la "
                           "tabla de consulta y le pega las columnas que encuentra. Es el BUSCARV de Excel, en una línea.")


def quien_manda_en_la_distancia(M, M_z, pos, nombres):
    """¿Qué variable decide la distancia de una canción (la fila `pos`) a todas las demás? Sin escalar y escalado."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.9))
    pares = [(M, "sin escalar: cada variable en su unidad"), (M_z, "escalado: todas en desviaciones estándar")]
    for ax, (mat, titulo) in zip(axes, pares):
        d2 = (np.asarray(mat, float) - np.asarray(mat, float)[pos]) ** 2     # diferencia al cuadrado con cada otra canción
        parte = 100 * d2.sum(axis=0) / d2.sum()                               # qué % del total aporta cada variable
        ax.barh(nombres, parte, color=[ROJO if p == parte.max() else AZUL for p in parte])
        ax.set_xlim(0, 120); ax.set_xlabel("% de la distancia total que aporta cada variable"); ax.set_title(titulo)
        for i, p in enumerate(parte):
            ax.text(p + 1.5, i, f"{p:.1f} %".replace(".", ","), va="center", fontsize=9)
    return _que_mirar(fig, "sin escalar, una sola variable (la de los números grandes) decide prácticamente el 100 % de quién es "
                           "'parecido' a quién; las demás ni se ven. Escalado, ninguna pasa del 30 %: todas opinan.")


def escalado_antes_despues(X, columnas):
    """Las mismas variables en sus unidades originales y después de estandarizar."""
    d = X[columnas].dropna(); z = (d - d.mean()) / d.std()
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    paneles = [(axes[0], d, "antes: cada variable en su unidad", "valor original"),
               (axes[1], z, "después: z = (x − media) / desviación", "desviaciones estándar")]
    for ax, datos, titulo, xl in paneles:
        ax.boxplot([datos[c] for c in columnas], vert=False, showfliers=False)
        ax.set_yticks(range(1, len(columnas) + 1)); ax.set_yticklabels(columnas); ax.set_title(titulo); ax.set_xlabel(xl)
    axes[1].axvline(0, color=ROJO, ls="--", lw=1)
    return _que_mirar(fig, f"a la izquierda solo se ve {columnas[-1]}: sus números son tan grandes que aplastan a las demás. "
                           "A la derecha todas quedan centradas en 0 con la misma dispersión; la forma de cada caja no cambia.")

def anatomia_grafica(serie, unidad_x="año", unidad_y="minutos"):
    """Las partes de una gráfica, señaladas sobre una gráfica real (una serie indexada por año)."""
    fig, ax = plt.subplots(figsize=(11, 5))
    serie.plot(ax=ax, marker="o", color=AZUL, lw=2, label="duración mediana (la serie)")
    ax.set_title("TÍTULO: la pregunta que responde la gráfica, con sus unidades", color=NARANJA, fontweight="bold")
    ax.set_xlabel(f"EJE X: qué hay a lo ancho ({unidad_x})", color=NARANJA, fontweight="bold")
    ax.set_ylabel(f"EJE Y: qué se mide ({unidad_y})", color=NARANJA, fontweight="bold")
    ax.grid(alpha=0.3); ax.legend(loc="upper right")
    flecha = dict(arrowstyle="->", color=NARANJA, lw=1.5)
    k = len(serie) // 2
    ax.annotate("marcador: un dato\n(un año, una mediana)", xy=(serie.index[-3], serie.iloc[-3]), xytext=(-235, -8),
                textcoords="offset points", color=NARANJA, fontsize=9, arrowprops=flecha)
    j = min(22, len(serie) - 2)
    ax.annotate("línea: une los datos\nde la misma serie", xy=(serie.index[j], serie.iloc[j]), xytext=(-75, 42),
                textcoords="offset points", color=NARANJA, fontsize=9, arrowprops=flecha)
    ax.annotate("leyenda: qué es cada serie\n(se necesita con dos o más)", xy=(0.86, 0.93), xycoords="axes fraction",
                xytext=(-250, -5), textcoords="offset points", color=NARANJA, fontsize=9, arrowprops=flecha)
    ax.annotate("rejilla: suave y de fondo,\npara leer valores", xy=(serie.index[5], serie.min() + 0.3 * (serie.max() - serie.min())),
                xytext=(10, -32), textcoords="offset points", color=NARANJA, fontsize=9, arrowprops=flecha)
    ax.annotate("marcas del eje (ticks)", xy=(0.12, -0.02), xycoords="axes fraction", xytext=(60, -28),
                textcoords="offset points", color=NARANJA, fontsize=9, arrowprops=flecha)
    return _que_mirar(fig, "la figura (fig) es el lienzo; el eje (ax) es el dibujo con sus dos ejes, título, leyenda y rejilla. "
                           "Todo lo que personaliza una gráfica es ponerle o quitarle una de estas partes.")


def pastel_vs_barras(serie):
    """La misma información como pastel (ilegible con muchas categorías) y como barras ordenadas."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1, 1.25]})
    serie.plot.pie(ax=axes[0], labels=None, startangle=90, counterclock=False)
    axes[0].set_ylabel(""); axes[0].set_title(f"pastel con {len(serie)} porciones: ¿cuál es la quinta más grande?")
    axes[0].legend(serie.index[:6].tolist() + ["…"], loc="center left", bbox_to_anchor=(0.95, 0.5), fontsize=8, frameon=False)
    serie.sort_values().plot.barh(ax=axes[1], color=AZUL)
    axes[1].set_title("las mismas categorías, en barras ordenadas"); axes[1].set_xlabel("canciones"); axes[1].set_ylabel("")
    axes[1].tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    return _que_mirar(fig, "el ojo compara mal ángulos y áreas, y bien longitudes alineadas. Con más de 4 o 5 categorías, "
                           "el pastel no responde ninguna pregunta; las barras ordenadas las responden todas.")

print("Figuras de hoy listas.")

Antes de empezar, el mapa. El curso entero sigue el pipeline de la Sesión 1; hoy vivimos en sus **dos primeras etapas** (datos y preprocesamiento), que son exactamente el territorio de los dos sombreros:

In [ ]:
fg.pipeline(["datos", "preprocesamiento"], modulo=1,
            subtitulo="HOY: las dos primeras etapas, con las herramientas que las hacen posibles (Python y pandas)");

In [ ]:
dos_sombreros();

## Lo que va a poder hacer al final (gancho)

### 🔍 Predice
Antes de correr la celda: de estos seis géneros —**EDM, latino, pop, R&B, rap, rock**—, ¿cuál cree que es el **más bailable** y cuál el **menos**? Anótelo.

**Su predicción:** más bailable ____ · menos bailable ____

In [ ]:
canciones = pd.read_csv(ARCHIVO)
promedio = canciones.groupby("playlist_genre")["danceability"].mean().sort_values()
promedio.plot.barh(color="#2E6FB7", title="¿Qué género es más bailable? (promedio de 'danceability', de 0 a 1)")
plt.xlabel("bailabilidad promedio"); plt.ylabel("")
plt.show()

Dos líneas de código, 32 833 canciones, una respuesta. No se preocupe si no entiende la celda: **al final de la clase cada palabra será suya** (`read_csv`, `groupby`, `mean`, `sort_values`, `plot`). Vamos por partes.

---
# Parte 1 — 🔧 El sombrero del ingeniero de datos

El ingeniero de datos no hace gráficos bonitos ni modelos: hace que los datos **lleguen, estén limpios y se pueda confiar en ellos**. Es el trabajo invisible que se lleva, en la vida real, la mayor parte del tiempo de cualquier proyecto de datos.

Para hacerlo necesitamos primero las **piezas del lenguaje**. Las vamos a ver con ejemplos de una sola canción (*Blinding Lights*, de The Weeknd, la más escuchada del mundo en 2020) y después con las 32 833 a la vez.

## 1. Colab y las piezas de Python (≈ 35 min)

### 1.1 Celdas, orden y su primer error

Un notebook es una lista de **celdas**. Las grises son código: se ejecutan con `Shift + Enter` y muestran su resultado debajo. Las blancas (como esta) son texto. Tres reglas:
1. Las celdas se ejecutan **en el orden en que usted las corre**, no en el que están escritas. El número `[3]` a la izquierda dice en qué turno se corrió.
2. Lo que una celda define (una variable, una función) **queda vivo** para las siguientes.
3. Si todo se enreda: *Entorno de ejecución → Reiniciar sesión* y vuelva a correr desde arriba. No se pierde nada de lo escrito.

In [ ]:
print("Hola, Univalle")
2 + 2

La celda mostró dos cosas: lo que pedimos con `print(...)` y, además, el valor de la **última línea** (por eso el `4`). Es una comodidad de los notebooks que usaremos todo el tiempo.

### 🔍 Predice — su primer error (a propósito)
La celda siguiente usa una variable llamada `genero` que **no hemos definido en ninguna parte**. ¿Qué cree que va a pasar? Ejecútela de todas formas. *(Si usó "Ejecutar todo", aquí se detiene: es intencional; siga ejecutando desde la celda que viene después.)*

In [ ]:
print(genero)   # 'genero' no existe todavía → esta celda da error A PROPÓSITO

Rojo, pero no pasó nada grave. Un error se lee **de abajo hacia arriba**: la **última línea** dice el tipo (`NameError`) y el mensaje (`name 'genero' is not defined`: la variable `genero` no está definida). Las líneas de arriba dicen en qué parte del código ocurrió. Eso es todo lo que hay que mirar el 90 % de las veces.

Lo arreglamos definiendo la variable y volviendo a ejecutar:

In [ ]:
genero = "reggaetón"
print(genero)

### 1.2 Variables y tipos: guardar cosas con nombre

Una **variable** es una etiqueta que le ponemos a un valor para usarlo después. Describamos una canción:

In [ ]:
cancion = "Blinding Lights"   # texto               → tipo str
artista = "The Weeknd"        # texto
duracion_ms = 201573          # número entero        → tipo int   (la duración viene en milisegundos)
bailabilidad = 0.513          # número con decimales → tipo float (de 0 a 1, lo calcula Spotify)
es_exito = True           # verdadero / falso    → tipo bool

type(cancion), type(duracion_ms), type(bailabilidad), type(es_exito)

### 🔍 Predice
`duracion_ms` son 201 573 milisegundos. ¿Cuántos **minutos** son, más o menos? ¿Y cuántos minutos y segundos exactos?

**Su predicción:** ____ min ____ s

In [ ]:
segundos = duracion_ms / 1000          # división normal → 201.573
minutos_decimal = duracion_ms / 60000  # 3.359...
minutos = duracion_ms // 60000         # división entera: cuántos minutos completos → 3
sobra_ms = duracion_ms % 60000         # el residuo: lo que no alcanzó a ser otro minuto → 21573
sobra_s = round(sobra_ms / 1000)       # redondeado → 22

print(segundos, minutos_decimal, minutos, sobra_s)
print(f"{cancion} dura {minutos} min y {sobra_s} s")   # f-string: texto con variables adentro, entre llaves

Las comparaciones producen un `bool` (verdadero o falso), y se combinan con `and`, `or` y `not`:

In [ ]:
print(bailabilidad > 0.5)                           # True: medio bailable (0,513); menos de lo que parece
print(duracion_ms < 180000)                         # False: dura más de 3 minutos
print(bailabilidad > 0.5 and duracion_ms < 180000)  # las dos a la vez → False
print(bailabilidad > 0.5 or duracion_ms < 180000)   # al menos una → True
print(artista == "The Weeknd", artista == "the weeknd")   # == compara; las mayúsculas importan

El texto tiene sus propias herramientas (se llaman **métodos** y se escriben con punto):

In [ ]:
print(artista.lower(), "|", artista.upper(), "|", cancion.title())
print(len(cancion), "caracteres (el espacio cuenta)")
print(cancion + " - " + artista)                # + pega textos
print("  Blinding Lights  ".strip())            # quita espacios al borde (muy útil con datos reales)
print(artista.replace("Weeknd", "Weekend"))     # el nombre artístico perdió una letra; aquí se la devolvemos

### 🧪 Juegue — dos errores clásicos
Quite el `#` de **una** de las líneas, ejecute, lea el error y vuelva a poner el `#`. Después mire la solución de cada uno en las líneas de abajo.

In [ ]:
# cancion + 2019            # TypeError: no se puede pegar texto con número…
print(cancion + " " + str(2019))              # …a menos que convierta el número a texto con str()

# float("0,803")             # ValueError: Python no entiende la coma decimal…
print(float("0,803".replace(",", ".")))       # …hay que cambiarla por punto. En Colombia esto pasa TODO el tiempo.

print(int("2019") + 1)                         # al revés: texto que parece número → número con int()

### 📘 Para entender — los cuatro tipos básicos

| Tipo | Qué guarda | Ejemplos | Qué se puede hacer |
|---|---|---|---|
| `int` | números enteros | `2019`, `200960`, `-3` | sumar, restar, multiplicar, dividir (`/`, `//`, `%`), comparar |
| `float` | números con decimales | `0.803`, `3.35`, `-6.5` | lo mismo que `int`; `round(x, 2)` redondea |
| `str` | texto, entre comillas | `"Blinding Lights"`, `'2019'` | pegar con `+`, `.lower()`, `.upper()`, `.strip()`, `.replace()`, `len()` |
| `bool` | verdadero o falso | `True`, `False` | combinar con `and`, `or`, `not`; es lo que devuelve una comparación |

**El tipo decide qué se puede hacer.** `"2019"` (texto) y `2019` (número) se ven iguales pero no son lo mismo: el primero no se puede sumar. La mitad de los problemas con datos reales son columnas que llegaron como texto cuando debían ser números (fechas, cifras con coma, códigos). `type(x)` dice qué es `x`; `int()`, `float()` y `str()` convierten.

**Punto, no coma.** Python (y pandas, y casi todo el software) usa **punto decimal**: `0.803`. En Colombia escribimos `0,803`, y los archivos de Excel colombianos vienen así. Cuando una columna de números llegue como texto, sospeche de la coma.

### 1.3 Listas: muchos valores en orden

Cuando hay varios valores del mismo tipo, van en una **lista**, entre corchetes y separados por coma. Estas son cinco de las canciones más populares del dataset:

In [ ]:
top5 = ["Blinding Lights", "ROXANNE", "Circles", "The Box", "Memories"]
popularidades = [98, 99, 98, 98, 98]

dibujar_lista(top5, nombre="top5");

### 🔍 Predice
¿Qué devuelve `top5[2]`? ¿Y `top5[-1]`? ¿Y `top5[1:3]` (¿incluye la posición 3 o no?)

**Su predicción:** `top5[2]` = ____ · `top5[-1]` = ____ · `top5[1:3]` = ____

In [ ]:
print(top5[0])        # el primero está en la posición 0
print(top5[2])        # la tercera posición
print(top5[-1])       # el último
print(top5[1:3])      # rebanada: desde la 1 hasta ANTES de la 3 → dos elementos
print(len(top5), "canciones")
print("Circles" in top5, "Bohemian Rhapsody" in top5)

Con números, las listas traen cuentas listas; y las listas se pueden modificar (`append` agrega al final):

In [ ]:
print(sum(popularidades), max(popularidades), min(popularidades))
print(sum(popularidades) / len(popularidades))   # el promedio, a mano
print(sorted(top5))                              # ordenada alfabéticamente (la original no cambia)

top5.append("Blinding Lights")                   # agrega al final (ahora son 6, pero el nombre se queda)
print(top5, len(top5))

### 🧪 Juegue
Pida `top5[6]`. Lea el error (`IndexError`): la lista tiene posiciones 0 a 5, la 6 no existe. Es el error más común cuando uno olvida que se cuenta desde cero.

In [ ]:
# top5[6]

### 1.4 Decidir y repetir: `if` y `for`

Un programa decide con `if` / `elif` / `else`. Fíjese en los **dos puntos** al final de cada condición y en la **sangría** (4 espacios): lo sangrado es "lo que pasa si se cumple".

In [ ]:
tempo = 171.0   # pulsaciones por minuto (bpm) de Blinding Lights

if tempo < 100:
    print("lenta")
elif tempo < 130:
    print("media")
else:
    print("rápida")

Y repite con `for`: "para cada elemento de la lista, haz esto". La variable `c` va tomando cada valor por turno:

In [ ]:
for c in top5:
    print("♪", c)

### 🔍 Predice
Los tempos de las cinco canciones del top son estos. Con la regla de arriba (lenta < 100 ≤ media < 130 ≤ rápida), ¿cuántas son **rápidas**?

**Su predicción:** ____ rápidas

In [ ]:
tempos = [171.0, 116.7, 120.0, 117.0, 91.0]   # Blinding Lights, ROXANNE, Circles, The Box, Memories

rapidas = 0                    # un contador que empieza en cero
for t in tempos:
    if t >= 130:
        rapidas = rapidas + 1  # cada vez que se cumple, suma uno
print("rápidas:", rapidas)

# Lo mismo, clasificando cada una:
for t in tempos:
    if t < 100:
        categoria = "lenta"
    elif t < 130:
        categoria = "media"
    else:
        categoria = "rápida"
    print(t, "→", categoria)

### 📘 Para entender — el `for` en una línea *(si hay tiempo)*
Python permite escribir "la lista de los `t` tales que…" en una sola línea. Se llama **comprensión de listas** y la verá mucho en código ajeno:

```python
[t for t in tempos if t >= 100]        # → [171.0, 116.7, 120.0, 117.0]
[c.upper() for c in top5]              # → ['BLINDING LIGHTS', 'ROXANNE', ...]
```
Es el mismo `for` de arriba, comprimido. Si le cuesta leerla, escríbala larga: funciona igual.

### 1.5 Diccionarios: un registro con etiquetas

Una lista guarda valores **por posición**; un **diccionario** los guarda **por nombre** (llave → valor), entre llaves `{}`. Es la forma natural de describir *una* canción completa:

In [ ]:
blinding = {"cancion": "Blinding Lights", "artista": "The Weeknd", "anio": 2019, "tempo": 171.0}

print(blinding["artista"])     # se accede por la llave, entre corchetes
blinding["genero"] = "pop"     # agregar una llave nueva es así de simple
print(blinding.keys())
dibujar_diccionario(blinding, nombre="blinding");

Ahora el paso clave de la sesión. Si tenemos **varios diccionarios con las mismas llaves** y los ponemos en una lista… eso **es una tabla**:

In [ ]:
roxanne = {"cancion": "ROXANNE", "artista": "Arizona Zervas", "anio": 2019, "tempo": 116.7}
bohemian = {"cancion": "Bohemian Rhapsody", "artista": "Queen", "anio": 1975, "tempo": 71.1}
tres = [blinding, roxanne, bohemian]       # una lista de diccionarios

for c in tres:
    print(c["cancion"], "→", c["anio"])

dibujar_tabla(tres);

### 🔍 Predice
`pd.DataFrame(tres)` convierte la lista de diccionarios en una tabla de pandas. ¿Cuántas filas y columnas tendrá? Ojo: `blinding` tiene una llave más que los otros dos (`genero`).

**Su predicción:** ____ filas × ____ columnas

In [ ]:
df_tres = pd.DataFrame(tres)
print(df_tres.shape)       # (filas, columnas)
df_tres

Eso de arriba es un **DataFrame**: la estructura central de pandas y del curso entero. Una fila es un registro (un diccionario); una columna es una variable; `NaN` es "no hay dato".

### 📘 Para entender — las cuatro estructuras de hoy

| Estructura | Guarda | Se accede con | Ejemplo |
|---|---|---|---|
| valor | una cosa | el nombre | `tempo = 101.1` |
| lista `[ ]` | varias cosas **en orden** | la posición: `lista[0]` | `top5 = ["Blinding Lights", "ROXANNE", ...]` |
| diccionario `{ }` | varias cosas **con nombre** | la llave: `dic["artista"]` | `blinding = {"artista": "The Weeknd", ...}` |
| DataFrame | una **tabla**: filas y columnas | la columna: `df["artista"]`; la fila: `df.loc[0]` | `pd.DataFrame(tres)` |

Todo lo que haremos con pandas se apoya en estas cuatro: una columna de un DataFrame se comporta como una lista; una fila, como un diccionario.

### 1.6 Funciones: empaquetar lo que se repite

Clasificamos tempos dos veces con el mismo `if`. Cuando algo se repite se escribe **una vez** como función (`def`), con un nombre, unos parámetros de entrada y un `return` que devuelve el resultado:

In [ ]:
def clasificar_tempo(bpm):
    """Devuelve 'lenta', 'media' o 'rápida' según los bpm."""   # el texto entre triples comillas documenta la función
    if bpm < 100:
        return "lenta"
    elif bpm < 130:
        return "media"
    return "rápida"


def a_minutos(ms):
    """Convierte milisegundos a minutos con dos decimales."""
    return round(ms / 60000, 2)


print(clasificar_tempo(101.1), clasificar_tempo(176), clasificar_tempo(60))
print(a_minutos(200960), "minutos")
print([clasificar_tempo(t) for t in tempos])     # la comprensión de listas aplicando la función a cada tempo

### 📘 Para entender — funciones, métodos e `import`
- Una **función** recibe parámetros y devuelve un resultado con `return`. `print()`, `len()`, `round()` y `type()` son funciones que ya vienen con Python.
- Un **método** es una función pegada a un objeto; se llama con punto: `cancion.upper()`, `top5.append(...)`, y en un rato `df.head()`.
- **`import`** trae funciones que otras personas escribieron. `import pandas as pd` carga pandas y le pone el apodo `pd`; por eso escribimos `pd.read_csv(...)`. Las librerías del curso: `pandas` (tablas), `numpy` (números), `matplotlib` (gráficos), `scikit-learn` (modelos).

Con esto ya tiene las piezas del lenguaje que usan los notebooks del curso. Lo que sigue es el trabajo de verdad.

## 2. pandas: el trabajo del ingeniero de datos (≈ 25 min)

### 2.1 Conseguir y entender: cargar la tabla y mirarla

`pd.read_csv` lee un archivo CSV (texto separado por comas, el formato universal de los datos) y devuelve un DataFrame. La variable se llama `df` por costumbre.

### 🔍 Predice
El archivo tiene una canción por fila y, según la descripción, 32 833 canciones. ¿Cuántas **columnas** cree que tiene (piense qué se puede saber de una canción)?

**Su predicción:** ____ filas × ____ columnas

In [ ]:
df = pd.read_csv(URL)
df.shape

In [ ]:
df.head(3)      # las primeras 3 filas (head = cabeza)

In [ ]:
df.info()       # una línea por columna: cuántos valores no vacíos tiene y de qué tipo es

`info()` es la radiografía de la tabla. Mire tres cosas: el número de filas (`entries`), que casi todas las columnas tienen 32 833 valores *non-null* (no vacíos) **menos tres**, y el tipo (`Dtype`): `object` es texto, `int64` números enteros, `float64` decimales. Son los mismos `str`, `int` y `float` de la sección 1.2, ahora por columnas.

`describe()` resume las columnas numéricas (promedio, mínimo, máximo, cuartiles):

In [ ]:
df.describe().round(2)

### 👀 Mire
Antes de limpiar, un ingeniero de datos se hace **la pregunta más importante de todas: ¿qué representa una fila?** Mire la columna `playlist_name` en `head(3)`: las tres primeras canciones están en la misma playlist ("Pop Remix"). Una fila no es una canción: es **una canción dentro de una playlist**. Guarde esa idea; en un momento nos va a cobrar.

### 2.2 Limpiar — el trabajo invisible

Vamos a hacer los cinco pasos de limpieza que aparecen en casi cualquier proyecto, y a llevar una **bitácora** de cuántas filas quedan después de cada uno (una lista de diccionarios, claro).

**(a) Nombres claros.** Los nombres están en inglés y algunos son largos (`track_album_release_date`). Los traducimos con un diccionario `nombre_viejo → nombre_nuevo`, en `snake_case` (minúsculas y guiones bajos, la convención del curso). Y botamos dos columnas de identificadores que no usaremos.

In [ ]:
bitacora = [{"paso": "archivo original", "filas": len(df)}]

nombres = {
    "track_name": "cancion", "track_artist": "artista", "track_popularity": "popularidad",
    "track_album_name": "album", "track_album_release_date": "fecha_lanzamiento",
    "playlist_name": "playlist", "playlist_genre": "genero", "playlist_subgenre": "subgenero",
    "danceability": "bailabilidad", "energy": "energia", "key": "tonalidad", "loudness": "volumen_db",
    "mode": "modo", "speechiness": "habla", "acousticness": "acustica", "instrumentalness": "instrumental",
    "liveness": "en_vivo", "valence": "positividad", "tempo": "tempo", "duration_ms": "duracion_ms",
}
df = df.rename(columns=nombres)
df = df.drop(columns=["track_album_id", "playlist_id"])
list(df.columns)

**(b) Tipos y fechas.** `fecha_lanzamiento` es texto. Miremos **cómo** viene antes de convertirla: la longitud del texto delata el formato.

### 🔍 Predice
Una fecha completa (`2019-06-14`) tiene 10 caracteres. ¿Cree que todas las fechas vienen completas?

In [ ]:
df["fecha_lanzamiento"].str.len().value_counts()

In [ ]:
df[df["fecha_lanzamiento"].str.len() < 10][["cancion", "artista", "fecha_lanzamiento"]].head(4)

Si convertimos a fecha con `pd.to_datetime`, pandas infiere el formato de la primera fila (`AAAA-MM-DD`) y **lo que no encaja queda como `NaT`** (*not a time*, el vacío de las fechas). Vea cuántas:

In [ ]:
fechas = pd.to_datetime(df["fecha_lanzamiento"], errors="coerce")   # errors="coerce": lo que no se pueda, vacío
print("fechas que no se pudieron convertir:", fechas.isna().sum())
fechas.head(3)

Para lo que necesitamos hoy basta el **año**, y ese sí está en todas: son los primeros cuatro caracteres del texto. `.str[:4]` es la rebanada de la sección 1.3 aplicada a toda la columna; `.astype(int)` la convierte a número. De paso, creamos la duración en minutos con la función que ya teníamos:

In [ ]:
df["anio"] = df["fecha_lanzamiento"].str[:4].astype(int)
df["duracion_min"] = df["duracion_ms"].apply(a_minutos)     # apply: aplica nuestra función fila por fila
df[["cancion", "fecha_lanzamiento", "anio", "duracion_ms", "duracion_min"]].head(3)

### 🔍 Predice
¿De qué año será la canción más antigua del dataset? ¿Y la más reciente?

**Su predicción:** ____ a ____

In [ ]:
print(df["anio"].min(), "a", df["anio"].max())
df["anio"].value_counts().sort_index().tail(8)      # cuántas canciones por año, los últimos 8 años

**(c) Duplicados.** Volvamos a la pregunta de qué es una fila.

### 🔍 Predice
¿Cuántas veces aparece **Blinding Lights** en la tabla? (Pista: la pregunta "¿qué representa una fila?")

In [ ]:
df[df["cancion"] == "Blinding Lights"][["cancion", "artista", "playlist", "genero", "popularidad"]]

Nueve filas, y no todas son la misma canción: ocho son *Blinding Lights* de The Weeknd en ocho playlists distintas, y la novena es otra canción que se llama igual. Si contáramos "canciones por artista" sin darnos cuenta, The Weeknd tendría ocho *Blinding Lights*; y si quitáramos duplicados **por nombre**, borraríamos una canción que no es la misma. La columna `track_id` identifica la canción de verdad; contemos cuántas filas repiten un `track_id` que ya apareció antes y quitémoslas con `drop_duplicates` (se queda con la **primera** aparición de cada canción):

In [ ]:
print("filas repetidas (misma canción en otra playlist):", df["track_id"].duplicated().sum())
df = df.drop_duplicates(subset="track_id")
bitacora.append({"paso": "quitar duplicados (track_id)", "filas": len(df)})
df.shape

> ⚠️ **Lo que acabamos de decidir, en voz alta.** Al quedarnos con la primera fila de cada canción, cada una queda con el género de la primera playlist donde apareció. Para *Blinding Lights* fue `pop` (razonable); para *Despacito* también fue `pop`, aunque es reggaetón. Es una simplificación razonable para hoy, pero **cambia la etiqueta de algunas canciones**, y eso le va a meter ruido al modelo del final. Un ingeniero de datos documenta estas decisiones; no las esconde.

**(d) Vacíos.** `info()` ya nos avisó: tres columnas tienen menos valores. `isna()` marca cada vacío con `True`; sumarlos los cuenta.

In [ ]:
df.isna().sum()

In [ ]:
df[df["cancion"].isna()][["track_id", "cancion", "artista", "popularidad", "playlist"]]

Cuatro filas sin nombre ni artista (y popularidad 0: canciones que ya no existen en Spotify). Son 4 de 28 356: las quitamos con `dropna`. Cuando los vacíos son muchos, o están en la columna que queremos predecir, borrar sale caro y hay que **rellenar**; eso lo vemos el miércoles.

In [ ]:
df = df.dropna(subset=["cancion", "artista"])
bitacora.append({"paso": "quitar filas sin nombre/artista", "filas": len(df)})
print("vacíos que quedan en toda la tabla:", df.isna().sum().sum())

**(e) Valores imposibles.** `describe()` mostró un mínimo de `duracion_ms` de **4 000** (¡4 segundos!) y un tempo mínimo de **0**. Ningún dato dijo "error": hay que saber qué es imposible en el dominio.

In [ ]:
df[(df["tempo"] == 0) | (df["duracion_ms"] < 30000)][["cancion", "artista", "tempo", "duracion_ms", "bailabilidad"]]

In [ ]:
df = df[(df["tempo"] > 0) & (df["duracion_ms"] >= 30000)]     # nos quedamos con las filas que cumplen las dos condiciones
bitacora.append({"paso": "quitar valores imposibles (tempo 0, < 30 s)", "filas": len(df)})
pd.DataFrame(bitacora)

Esa tabla es la **bitácora del ingeniero**: de 32 833 filas a 28 350 canciones únicas, y cada paso explicado. En un informe real se escribe exactamente así.

### 📘 Para entender — los cinco pasos de limpieza

| Paso | Pregunta que responde | En pandas |
|---|---|---|
| Nombres | ¿se entiende cada columna sin mirar un manual? | `rename(columns={...})`, `drop(columns=[...])` |
| Tipos y fechas | ¿los números son números y las fechas son fechas? | `info()`, `astype(int)`, `pd.to_datetime(..., errors="coerce")`, `.str[:4]` |
| Duplicados | ¿qué representa una fila? ¿hay filas que son "la misma cosa"? | `duplicated().sum()`, `drop_duplicates(subset=...)` |
| Vacíos | ¿dónde falta información y por qué? | `isna().sum()`, `dropna(subset=...)`, (`fillna`, el miércoles) |
| Valores imposibles | ¿qué no puede pasar en este dominio? | `describe()`, filtros con `>`, `<`, `==` |

El orden importa poco; lo que importa es **hacerlos todos y anotar cuántas filas se fueron en cada uno**.

### 2.3 Transformar: seleccionar, filtrar, ordenar, contar, agrupar

Cinco operaciones resuelven el 80 % de las preguntas sobre una tabla. **Seleccionar** columnas:

In [ ]:
df["artista"].head(3)                                   # una columna → se comporta como una lista (se llama Series)

In [ ]:
df[["cancion", "artista", "anio", "popularidad"]].head(3)   # varias columnas → lista de nombres, doble corchete

**Filtrar** filas con una condición. Es el `if` de la sección 1.4 aplicado a todas las filas a la vez: la condición produce una columna de `True`/`False` y `df[condición]` se queda con las `True`:

In [ ]:
juanes = df[df["artista"] == "Juanes"]
print(len(juanes), "canciones de Juanes")
juanes[["cancion", "anio", "popularidad", "genero"]].sort_values("popularidad", ascending=False).head(5)

Las condiciones se combinan con `&` (y) y `|` (o), **cada una entre paréntesis**. Y para buscar dentro del texto está `.str.contains`:

In [ ]:
recientes_latinas = df[(df["genero"] == "latin") & (df["anio"] >= 2019)]
print(len(recientes_latinas), "canciones latinas de 2019 en adelante")

balvin = df[df["artista"].str.contains("Balvin", case=False, regex=False)]   # case=False: sin importar mayúsculas
print(len(balvin), "canciones donde el artista contiene 'Balvin'")
balvin[["cancion", "artista", "anio", "popularidad"]].sort_values("popularidad", ascending=False).head(5)

### 🧪 Juegue
Cambie `"Balvin"` por su artista favorito (Maluma, Juanes, Carlos Vives, Queen, Bad Bunny…) y vuelva a correr. Si sale `0 canciones`, pruebe con parte del nombre.

**Ordenar** y **contar**:

### 🔍 Predice
¿Estará *Blinding Lights* en el top 10 de popularidad? ¿Qué artista tendrá más canciones distintas en todo el dataset?

In [ ]:
df.sort_values("popularidad", ascending=False)[["cancion", "artista", "genero", "popularidad", "anio"]].head(10)

In [ ]:
df["artista"].value_counts().head(10)    # cuántas filas (canciones) por artista, de mayor a menor

In [ ]:
df["genero"].value_counts()               # el contador + for + if de la sección 1.4, en una línea

**Agrupar**: la operación más poderosa de pandas. `groupby("genero")` parte la tabla en un grupo por género; después pedimos una cuenta por grupo (`mean`, `median`, `count`, `max`…). Es exactamente la celda del gancho:

In [ ]:
df.groupby("genero")["bailabilidad"].mean().sort_values(ascending=False).round(3)

### 🔍 Predice
Ahora varias columnas a la vez. Antes de correr: ¿qué género tendrá la **positividad** (qué tan "alegre" suena) más alta? ¿Cuál tendrá más **habla** (proporción de palabras habladas)? ¿Cuál será el más **largo**?

**Su predicción:** más positivo ____ · más hablado ____ · más largo ____

In [ ]:
(df.groupby("genero")[["bailabilidad", "energia", "positividad", "volumen_db", "habla", "duracion_min", "anio"]]
   .mean()
   .round(2)
   .sort_values("bailabilidad", ascending=False))

### 2.4 Juntar tablas: `merge`

Casi nunca toda la información está en una sola tabla. Tenemos los géneros como códigos cortos (`edm`, `r&b`…) y, en otro archivo, una tabla pequeña con el nombre en español y una descripción. `merge` **busca** cada código en la segunda tabla y trae sus columnas: el BUSCARV de Excel.

In [ ]:
generos = pd.read_csv(ARCHIVO_GENEROS)
generos

In [ ]:
ejemplo = df.drop_duplicates("genero")[["artista", "genero", "bailabilidad"]].head(4)   # 4 filas de géneros distintos, para el esquema
dibujar_merge(ejemplo, generos[["genero", "nombre"]], llave="genero");

In [ ]:
antes = df.shape
df = df.merge(generos[["genero", "nombre"]], on="genero", how="left")   # how="left": conservar todas las filas de df
print(antes, "→", df.shape)
df[["cancion", "genero", "nombre"]].sample(5, random_state=1)

Dos cosas para fijarse: la tabla **no cambió de filas** (todas encontraron su género) y **ganó una columna**. Si una llave no existe en la tabla de consulta, `how="left"` deja `NaN` en vez de borrar la fila. Revisar que las filas se conserven después de un `merge` es un hábito que ahorra muchos dolores de cabeza.

Un pariente simple de `merge`, para traducir una columna con un diccionario (¡otra vez!): `.map`. La tonalidad viene como número (0 = Do, 1 = Do♯, …, 11 = Si) y el modo como 1/0:

In [ ]:
NOTAS = {0: "Do", 1: "Do#", 2: "Re", 3: "Re#", 4: "Mi", 5: "Fa", 6: "Fa#", 7: "Sol", 8: "Sol#", 9: "La", 10: "La#", 11: "Si"}
df["nota"] = df["tonalidad"].map(NOTAS)
df["modo_nombre"] = df["modo"].map({1: "mayor", 0: "menor"})
df[["cancion", "tonalidad", "nota", "modo", "modo_nombre"]].head(4)

### 🔍 Predice *(curiosidad)*
¿Qué nota será la tonalidad más común entre 28 mil canciones? ¿Y la más rara? ¿Habrá más canciones en modo mayor o menor?

In [ ]:
print(df["nota"].value_counts().head(3), "\n...\n", df["nota"].value_counts().tail(2), sep="")
df["modo_nombre"].value_counts(normalize=True).round(2)    # normalize=True: proporciones en vez de conteos

### ✅ La entrega del ingeniero
Lo que recibimos y lo que entregamos, en la bitácora. Y la tabla limpia se guarda en un archivo nuevo (nunca se sobreescribe el original):

In [ ]:
pd.DataFrame(bitacora)

In [ ]:
df.to_csv("spotify_limpio.csv", index=False)     # index=False: no guardar el número de fila como columna
print("Entregado: spotify_limpio.csv con", df.shape[0], "canciones y", df.shape[1], "columnas")

### ⚡ Reto 1 (opcional, sombrero de ingeniero)
Con lo visto hasta aquí, responda en la celda de abajo:
1. ¿Cuál es la canción **más larga** y cuál la **más corta** después de la limpieza? (`sort_values` por `duracion_min`).
2. ¿En qué **año** se lanzaron más canciones de las que hay en el dataset? (`value_counts` sobre `anio`).
3. **Difícil:** ¿qué artista aparece en **más géneros distintos**? Pista: `df.groupby("artista")["genero"].nunique().sort_values(ascending=False).head()`.

<details><summary>Pistas</summary>

1. `df.sort_values("duracion_min").head(1)` y `.tail(1)` (o `ascending=False`).
2. `df["anio"].value_counts().head(1)`.
3. `nunique()` cuenta valores distintos dentro de cada grupo. Compare el resultado con lo que sabe de esos artistas.
</details>

In [ ]:
# Espacio para el reto 1


---
> ☕ **Pausa de 5 minutos.** Al volver nos cambiamos de sombrero.
---

# Parte 2 — 🔬 El sombrero del científico de datos

El científico de datos recibe la tabla confiable y hace tres cosas: **pregunta**, **responde con gráficos y números**, y **prepara la tabla para que un modelo aprenda**. Hoy, además, entrenamos uno al final.

## 3. Taller de gráficas: de la pregunta al dibujo (≈ 27 min)

Una gráfica es **una pregunta hecha dibujo**. En Python se construye siempre con la misma receta de cuatro pasos, y la vamos a repetir hasta que salga sola:

| Paso | Qué se hace | Con qué |
|---|---|---|
| 1. Datos | filtrar o agrupar hasta tener una tabla **pequeña** con exactamente lo que se va a dibujar | `df[...]`, `groupby`, `value_counts` |
| 2. Tipo | elegir el tipo de gráfica según la pregunta | `.plot()`, `.plot.bar()`, `.plot.barh()`, `.plot.hist()`, `df.boxplot()`, `df.plot.scatter()` |
| 3. Títulos | título que diga la pregunta, ejes con unidades | `plt.title()`, `plt.xlabel()`, `plt.ylabel()` |
| 4. Ajustes | tamaño, color, rejilla, leyenda; y mostrar | `figsize`, `color`, `plt.grid()`, `plt.legend()`, `plt.show()` |

pandas dibuja con **matplotlib** por debajo: `df.plot(...)` es un atajo, y `plt.` (matplotlib) es donde se ajustan los detalles. Primero, las partes de una gráfica, señaladas sobre una de verdad:

In [ ]:
por_anio = df.groupby("anio")["duracion_min"].median().loc[1990:2020]   # la gráfica de ejemplo: duración típica por año
anatomia_grafica(por_anio);

### 3.1 La misma gráfica, paso a paso (de fea a presentable)

### 🔍 Predice
Comparada con una canción típica del año 2000, ¿una de 2019 es más larga, más corta o igual? ¿Por cuánto?

**Su predicción:** ____

**Paso 1 — datos.** Agrupar por año y quedarnos con la mediana (la canción "del medio" de cada año):

In [ ]:
por_anio = df.groupby("anio")["duracion_min"].median().loc[1990:2020]
por_anio.tail(5)

**Paso 2 — tipo.** Una cantidad que cambia en el tiempo pide una **línea**. Con una sola palabra ya hay gráfica… fea, pero gráfica:

In [ ]:
por_anio.plot()
plt.show()

**Paso 3 — títulos y ejes.** Sin título ni unidades, nadie sabe qué está viendo (ni usted, dentro de un mes):

In [ ]:
por_anio.plot()
plt.title("Duración típica (mediana) de las canciones por año de lanzamiento")
plt.xlabel("año"); plt.ylabel("minutos")
plt.show()

**Paso 4 — ajustes.** Tamaño, color, marcadores y rejilla suave. Y una decisión que hay que tomar a conciencia: el eje Y **no empieza en cero** (pandas lo recorta solo), lo que hace ver la caída más dramática de lo que es. En una línea de tiempo es aceptable **si se declara**; en barras nunca se recorta.

In [ ]:
por_anio.plot(marker="o", color="#2E6FB7", figsize=(9, 3.8))
plt.title("Duración típica (mediana) de las canciones por año de lanzamiento")
plt.xlabel("año"); plt.ylabel("minutos (eje desde 3, no desde 0)")
plt.grid(alpha=0.3)
plt.show()
print(por_anio.loc[[1990, 2000, 2010, 2019]].round(2))

### 🧪 Juegue
Cambie `"duracion_min"` por `"popularidad"` o por `"bailabilidad"`, y `.median()` por `.mean()`. Cambie el color (`"#C4463A"`, `"green"`, `"gray"`). Quite el `marker`. Cada cambio es una celda que se vuelve a correr: así se aprende a graficar.

### 3.2 Galería de recetas: un tipo de gráfica por pregunta

Cada celda sigue la misma receta de cuatro pasos. Guarde esta sección como plantilla: en su proyecto va a copiar y pegar de aquí.

**Barras horizontales — comparar una cantidad entre categorías.** *¿Qué género suena más fuerte?* El volumen (`volumen_db`) viene en decibeles **negativos**: 0 es el máximo posible y −10 es más suave que −5.

In [ ]:
volumen = df.groupby("nombre")["volumen_db"].mean().sort_values()          # 1) datos, ordenados para que la barra más larga quede arriba
volumen.plot.barh(color="#2E6FB7", figsize=(8, 3.6))                        # 2) tipo
plt.title("Volumen promedio por género (dB; más cerca de 0 = más fuerte)")  # 3) títulos
plt.xlabel("dB"); plt.ylabel("")
plt.grid(axis="x", alpha=0.3); plt.show()                                     # 4) ajustes

*Qué mirar: la electrónica de baile está casi 3 dB por encima del R&B; como los decibeles son logarítmicos, 3 dB es cerca del doble de potencia sonora. Las barras van ordenadas: una barra desordenada obliga al lector a ordenar con los ojos.*

**Barras verticales con etiquetas rotadas — conteos por categoría ordenada.** *¿De qué años son las canciones del dataset?*

In [ ]:
por_anio_n = df["anio"].value_counts().sort_index().loc[2005:2020]       # 1) datos: cuántas canciones por año
por_anio_n.plot.bar(color="#2E6FB7", figsize=(9, 3.4), rot=45)              # 2) tipo (rot: gira las etiquetas del eje X)
plt.title("Canciones del dataset por año de lanzamiento (2005–2020)")      # 3) títulos
plt.xlabel("año"); plt.ylabel("canciones")
plt.grid(axis="y", alpha=0.3); plt.show()                                   # 4) ajustes

*Qué mirar: 2019 tiene más de siete mil canciones (ya sin duplicados) y 2020 apenas empezaba cuando se armó el dataset (enero). Dos sesgos a declarar antes de concluir cualquier cosa "por año".*

**Histograma — cómo se reparten los valores de una variable.** *¿Cuántas canciones hay en cada nivel de bailabilidad?* `bins` es el número de franjas.

In [ ]:
df["bailabilidad"].plot.hist(bins=30, color="#2E6FB7", edgecolor="white", figsize=(8, 3.4))
plt.title("¿Cuántas canciones hay en cada nivel de bailabilidad?"); plt.xlabel("bailabilidad (0 a 1)"); plt.ylabel("canciones")
plt.show()

**Dos histogramas superpuestos — comparar la distribución de dos grupos.** Dos llamadas seguidas dibujan en la **misma** figura; `alpha` las hace transparentes y `label` + `plt.legend()` dicen cuál es cuál.

### 🔍 Predice
¿Se van a superponer mucho las dos campanas (rock y latino) o van a quedar separadas?

In [ ]:
df[df["genero"] == "rock"]["bailabilidad"].plot.hist(bins=30, alpha=0.6, color="#2E6FB7", label="rock", figsize=(8, 3.6))
df[df["genero"] == "latin"]["bailabilidad"].plot.hist(bins=30, alpha=0.6, color="#C4463A", label="latino")
plt.title("Bailabilidad: rock vs latino"); plt.xlabel("bailabilidad (0 a 1)"); plt.ylabel("canciones")
plt.legend(); plt.show()

**Cajas por grupo — la distribución de todos los grupos a la vez.** La línea del medio es la mediana; la caja va del 25 % al 75 % de las canciones; los puntos sueltos son casos atípicos.

### 🔍 Predice
¿Qué género tendrá la caja **más abajo**? ¿Y la **más ancha** (más variedad)?

In [ ]:
df.boxplot(column="bailabilidad", by="genero", figsize=(10, 4), grid=False)
plt.suptitle(""); plt.title("Bailabilidad por género"); plt.xlabel(""); plt.ylabel("bailabilidad")
plt.show()

**Dispersión — ¿dos variables van juntas?** Cada canción es un punto. Con 28 mil puntos hay que hacerlos pequeños (`s`) y transparentes (`alpha`) para ver dónde se amontonan.

### 🔍 Predice
*¿Lo que suena fuerte se siente enérgico?* ¿La nube de puntos sube hacia la derecha, baja o es un bloque sin forma?

In [ ]:
df.plot.scatter(x="volumen_db", y="energia", alpha=0.08, s=5, color="#2E6FB7", figsize=(6.5, 5))
plt.title("Cada punto es una canción: volumen vs energía"); plt.xlabel("volumen (dB)"); plt.ylabel("energía (0 a 1)")
plt.show()

El ojo engaña con tantos puntos. El número que resume "qué tanto van juntas" dos variables es la **correlación**: va de −1 (cuando una sube la otra baja) a +1 (suben juntas); 0 es "no hay relación lineal". `corr()` la calcula para todas las parejas a la vez:

In [ ]:
df[["bailabilidad", "energia", "positividad", "volumen_db", "tempo", "acustica", "duracion_min"]].corr().round(2)

**Varias series en una línea de tiempo — comparar grupos a lo largo del tiempo.** Otra vez: dos llamadas seguidas, cada una con su `label`, y una leyenda.

In [ ]:
rock = df[df["genero"] == "rock"].groupby("anio")["duracion_min"].median().loc[1995:2020]   # 1) datos, un grupo
rap = df[df["genero"] == "rap"].groupby("anio")["duracion_min"].median().loc[1995:2020]      #    y el otro
rock.plot(label="rock", color="#2E6FB7", marker="o", figsize=(9, 3.8))                        # 2) tipo, una serie
rap.plot(label="rap", color="#C4463A", marker="o")                                            #    y la otra, en la misma figura
plt.title("Duración típica por año: rock vs rap")                                              # 3) títulos
plt.xlabel("año"); plt.ylabel("minutos")
plt.legend(); plt.grid(alpha=0.3); plt.show()                                                  # 4) leyenda y ajustes

**Dos gráficas lado a lado — `subplots`.** Cuando quiere comparar dos gráficas, se crea una figura con varios "ejes" y se le dice a cada gráfica en cuál dibujarse (`ax=`). Es la forma en que están hechas todas las figuras del curso.

In [ ]:
por_genero = df.groupby("nombre")[["bailabilidad", "positividad"]].mean()        # 1) datos para las dos
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))                                   # una figura, 1 fila × 2 ejes
por_genero["bailabilidad"].sort_values().plot.barh(ax=axes[0], color="#2E6FB7")   # 2) tipo, en el eje izquierdo
por_genero["positividad"].sort_values().plot.barh(ax=axes[1], color="#2C8C5B")    #    y en el derecho
axes[0].set_title("Bailabilidad promedio por género"); axes[0].set_xlabel("0 a 1"); axes[0].set_ylabel("")   # 3) títulos, por eje
axes[1].set_title("Positividad promedio por género"); axes[1].set_xlabel("0 a 1"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()                                                      # 4) tight_layout: que no se pisen

*Qué mirar: con `ax=` los títulos se ponen con `axes[0].set_title(...)` en vez de `plt.title(...)`. Es la misma receta, solo cambia a quién se le habla.*

**Guardar una gráfica** (para el informe, la presentación o el proyecto): `plt.savefig` va **antes** de `plt.show()`. En Colab el archivo aparece en el panel de la izquierda (📁), desde donde se descarga.

In [ ]:
volumen.plot.barh(color="#2E6FB7", figsize=(8, 3.6))
plt.title("Volumen promedio por género (dB)"); plt.xlabel("dB"); plt.ylabel("")
plt.savefig("volumen_por_genero.png", dpi=150, bbox_inches="tight")   # dpi: nitidez; bbox_inches: que no se corten las etiquetas
plt.show()
print("Guardada: volumen_por_genero.png (mírela en el panel de archivos de Colab)")
# Para descargarla al computador desde Colab: from google.colab import files; files.download("volumen_por_genero.png")

### 📘 Para entender — qué gráfica para qué pregunta, y cinco errores que se ven todos los días

| Pregunta | Gráfica | En pandas |
|---|---|---|
| ¿Cómo cambia algo en el tiempo? | línea (varias series: varias llamadas + `legend`) | `serie.plot()` |
| ¿Cómo se compara una cantidad entre categorías? | barras **ordenadas**; horizontales si los nombres son largos | `.plot.barh()` / `.plot.bar(rot=45)` |
| ¿Cómo se reparten los valores de una variable? | histograma; cajas si es por grupo | `.plot.hist(bins=30)`; `df.boxplot(column=, by=)` |
| ¿Dos variables van juntas? | dispersión + correlación | `df.plot.scatter(x=, y=, alpha=)`; `df[cols].corr()` |
| ¿Qué parte del total es cada categoría? | barras (el pastel solo con 2–4 categorías) | `.plot.barh()` |

| Error | Por qué engaña | Qué hacer |
|---|---|---|
| Pastel con muchas porciones | el ojo compara mal ángulos | barras ordenadas (vea la figura de abajo) |
| Eje Y recortado en barras | una diferencia pequeña se ve enorme | barras siempre desde cero; en líneas, declarar el recorte |
| Dos ejes Y en la misma gráfica | se puede "demostrar" cualquier relación moviendo las escalas | dos gráficas lado a lado (`subplots`) |
| Arcoíris de colores | el color no significa nada y cansa | un color; dos o tres si hay grupos, siempre los mismos |
| Sin título ni unidades | nadie sabe qué está viendo | la pregunta en el título; unidades en los ejes |

**La regla de fondo:** correlación no es causa. Que lo enérgico suene fuerte no dice qué causa qué, ni si una tercera cosa (el género, la época) explica las dos.

In [ ]:
pastel_vs_barras(df["subgenero"].value_counts());

### ✏️ Su turno — construya su propia gráfica
La celda de abajo ya responde una pregunta con la receta de cuatro pasos. **Cámbiela** para responder otra. Tres sugerencias, de más fácil a más difícil:

1. ¿Qué género tiene las canciones **más populares** en promedio? (cambie la columna).
2. ¿Cómo se reparte el **tempo**? ¿Hay un pico? (cambie el tipo a histograma: `df["tempo"].plot.hist(bins=40)`).
3. ¿Las canciones más **bailables** son más **positivas**? Dibújelo solo para el rock y solo para el latino, en dos gráficas lado a lado (`subplots` + `plot.scatter(ax=...)`).

<details><summary>Pistas</summary>

- Paso 1 cambia la columna o el filtro; paso 2 cambia el tipo; los pasos 3 y 4 casi nunca cambian.
- Para la 3: `df[df["genero"] == "rock"].plot.scatter(x="bailabilidad", y="positividad", alpha=0.1, s=5, ax=axes[0])`.
- Si algo falla, lea el último renglón del error: casi siempre es un nombre de columna mal escrito (`KeyError`).
</details>

In [ ]:
# ✏️ Su turno. Esta celda funciona tal cual; modifíquela para responder su pregunta.
# Pregunta: ¿qué género tiene las canciones más populares en promedio?

datos_grafica = df.groupby("nombre")["popularidad"].mean().sort_values()     # 1) datos: cambie la columna o el filtro
datos_grafica.plot.barh(color="#2E6FB7", figsize=(8, 3.5))                   # 2) tipo: barh, bar, hist, scatter, plot
plt.title("Popularidad promedio por género (0 a 100)")                       # 3) título = la pregunta
plt.xlabel("popularidad promedio"); plt.ylabel("")
plt.grid(axis="x", alpha=0.3); plt.show()                                     # 4) ajustes y mostrar

### 📘 seaborn: colorear por categoría en una línea *(si hay tiempo)*
**seaborn** es una librería construida sobre matplotlib que hace en una llamada lo que en pandas toma varias: por ejemplo, un diagrama de dispersión con un color por género (`hue`). Viene instalada en Colab. Úsela cuando necesite "una serie por categoría"; para lo demás, la receta de pandas basta.

In [ ]:
import seaborn as sns

PALETA_GENEROS = {"edm": "#2E6FB7", "latin": "#C4463A", "pop": "#2C8C5B", "r&b": "#D97C1F", "rap": "#7A4DB8", "rock": "#8C8C8C"}
muestra = df.sample(3000, random_state=1)                                   # 3 000 canciones al azar: con 28 mil no se ve nada
plt.figure(figsize=(7.5, 5))
sns.scatterplot(data=muestra, x="energia", y="acustica", hue="genero", palette=PALETA_GENEROS, alpha=0.6, s=14)
plt.title("Energía vs acústica, un color por género (muestra de 3 000 canciones)"); plt.xlabel("energía (0 a 1)"); plt.ylabel("acústica (0 a 1)")
plt.legend(title="género", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.show()

*Qué mirar: la relación negativa energía–acústica (corr −0,55) y dónde vive cada género: el EDM arriba a la izquierda (enérgico, nada acústico), el R&B más abajo. Cada género tiene **siempre el mismo color** en todas las figuras de hoy; ese es el hábito que hace legible un informe.*

## 4. Preparar la tabla para un modelo (≈ 16 min)

Un modelo de scikit-learn necesita dos cosas: **X**, una tabla **solo de números y sin vacíos** (las variables de entrada), y **y**, la columna que queremos predecir. Nuestra tabla todavía tiene texto, categorías disfrazadas de número y variables en escalas incomparables. Arreglar eso es el **preprocesamiento**.

**La pregunta de hoy:** ¿puede una máquina adivinar el **género** de una canción *oyendo solo los números* (bailabilidad, energía, tempo, volumen, duración…)? Entonces `y = genero` (6 clases) y `X` = las características de audio.

### 🔍 Predice
Un "modelo" tonto que siempre responde el género más frecuente, ¿qué porcentaje acertaría? (Pista: `value_counts(normalize=True)`).

**Su predicción:** ____ %

In [ ]:
y = df["genero"]
y.value_counts(normalize=True).round(3)

### 4.1 Categorías → números: *one-hot* (y una categoría disfrazada)

La tonalidad viene como número (0 a 11) pero **es una categoría**: Re (2) no es "el doble" de Do♯ (1), y Si (11) no es "mayor" que Do (0). Si se la damos al modelo como número, le estamos inventando un orden que no existe. La solución es la misma de la Sesión 1: **one-hot**, una columna de 0/1 por categoría.

In [ ]:
fg.one_hot_ejemplo(df, "nota");

### 🔍 Predice
Vamos a armar `X` con 11 columnas numéricas + la tonalidad en one-hot + el modo (que ya es 0/1). ¿Cuántas columnas tendrá `X`?

**Su predicción:** ____ columnas

In [ ]:
numericas = ["bailabilidad", "energia", "volumen_db", "habla", "acustica", "instrumental",
             "en_vivo", "positividad", "tempo", "duracion_min", "anio"]
tonalidad_oh = pd.get_dummies(df["nota"], prefix="ton", dtype=int)      # 12 columnas de 0/1
X = pd.concat([df[numericas], tonalidad_oh, df[["modo"]]], axis=1)      # concat con axis=1: pegar columnas lado a lado

print("X:", X.shape, "  vacíos en X:", X.isna().sum().sum())
X.head(3)

### 4.2 Escalas distintas → estandarizar

Mire los rangos de `X`: la bailabilidad va de 0 a 1, el tempo de 50 a 240, el año de 1957 a 2020 y la duración en milisegundos (si la hubiéramos dejado) de 30 000 a 517 000. Para muchos modelos eso es un problema, y la forma más clara de verlo es con una pregunta concreta:

### ¿Qué canción se parece más a Blinding Lights?

"Parecida" = cercana en todas las características a la vez (la **distancia** entre dos filas, como en un mapa pero con 8 ejes). Calculemos la canción más cercana a *Blinding Lights* de dos maneras: con los números **tal como vienen** (incluida la duración en milisegundos) y **estandarizados**.

### 🔍 Predice
Sin escalar, ¿qué variable cree que va a decidir quién es "la más parecida"?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

rasgos = ["bailabilidad", "energia", "volumen_db", "habla", "acustica", "positividad", "tempo", "duracion_ms"]
M = df[rasgos].to_numpy()                                   # la tabla como matriz de números
M_z = StandardScaler().fit_transform(M)                     # la misma matriz, estandarizada
i_bl = df.index[(df["cancion"] == "Blinding Lights") & (df["artista"] == "The Weeknd")][0]   # la de The Weeknd, no la homónima
pos = df.index.get_loc(i_bl)

def mas_parecidas(matriz, k=3):
    """Las k filas más cercanas a Blinding Lights según la matriz que se le pase."""
    vecinos = NearestNeighbors(n_neighbors=k + 1).fit(matriz)
    _, idx = vecinos.kneighbors(matriz[[pos]])
    return df.iloc[idx[0][1:]][["cancion", "artista", "nombre", "duracion_ms", "bailabilidad", "tempo"]]

print("SIN escalar — las 3 más parecidas a Blinding Lights:")
display(mas_parecidas(M))
print("ESCALADO — las 3 más parecidas a Blinding Lights:")
display(mas_parecidas(M_z))

Mire las dos tablas: sin escalar, las tres "más parecidas" duran **201 560, 201 561 y 201 579 ms** (*Blinding Lights* dura 201 573) : la duración decide y el tempo (la segunda variable con números grandes) desempata; la bailabilidad, la energía y lo demás ni se miran. Escaladas, las vecinas comparten el estilo completo (rápidas, enérgicas, nada acústicas) aunque duren distinto. ¿Por qué? Porque la distancia suma las diferencias de todas las variables, y una diferencia de 2 000 milisegundos pesa infinitamente más que una de 0,2 en bailabilidad. La figura lo mide: qué porcentaje de la distancia de *Blinding Lights* a *todas* las demás canciones aporta cada variable.

In [ ]:
quien_manda_en_la_distancia(M, M_z, pos, rasgos);

### 📘 Para entender — estandarización (z-score)

**Qué es.** Restarle a cada columna su media y dividir por su desviación estándar:

$$z = \frac{x - \text{media}}{\text{desviación estándar}}$$

Después, **todas** las columnas tienen media 0 y desviación 1, y un valor se lee igual en cualquiera de ellas: $z = 2$ significa "dos desviaciones por encima del promedio", sea tempo, duración o volumen. La forma de la distribución no cambia; solo la regla con que se mide.

**Para qué sirve.** Para que ninguna variable mande solo por tener números grandes. La necesitan los modelos que miden **distancias** o suman variables con pesos: k-NN, K-means, PCA, regresión logística y lineal (sobre todo con regularización), redes neuronales. **No** la necesitan los árboles y bosques, que preguntan "¿tempo ≤ 120?" y les da igual la unidad (el miércoles lo verá con el árbol de Medellín).

**Alternativa.** Min-max: llevar cada columna al rango 0–1 con $(x - \min)/(\max - \min)$. Útil cuando se quieren valores acotados; sensible a los valores extremos.

<details><summary><b>Para quien quiera más:</b> por qué se ajusta solo con entrenamiento</summary>

La media y la desviación se **aprenden** de los datos, igual que un modelo. Si se calculan con toda la tabla, la parte de prueba "contamina" lo que se aprendió. Por eso en la celda siguiente el escalador hace `fit` solo con `X_train` y después `transform` en los dos. Es una forma suave de la trampa que verán el miércoles y la otra semana (S3): nada que se aprenda de los datos puede ver la prueba.
</details>

Ahora sí, el preprocesamiento como se hace de verdad: separar entrenamiento y prueba (la regla de la S1) y ajustar el escalador **solo con entrenamiento**.

### 🔍 Predice
Después de estandarizar, ¿cuánto darán la **media** y la **desviación estándar** de la columna `tempo` en entrenamiento? ¿Y en prueba: exactamente lo mismo o parecido?

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

escalador = StandardScaler().fit(X_train[numericas])        # aprende media y desviación SOLO de entrenamiento
X_train_esc = X_train.copy(); X_test_esc = X_test.copy()
X_train_esc[numericas] = escalador.transform(X_train[numericas])
X_test_esc[numericas] = escalador.transform(X_test[numericas])

print("Entrenamiento:", X_train.shape[0], "canciones   Prueba:", X_test.shape[0])
print("tempo en entrenamiento → media", round(X_train_esc["tempo"].mean(), 3), " desviación", round(X_train_esc["tempo"].std(), 3))
print("tempo en prueba         → media", round(X_test_esc["tempo"].mean(), 3), " desviación", round(X_test_esc["tempo"].std(), 3))

In [ ]:
escalado_antes_despues(X_train, ["bailabilidad", "volumen_db", "tempo", "anio"]);

`X_train_esc` es la tabla que "come" un modelo: **solo números, sin vacíos, categorías en one-hot, numéricas en la misma escala**. Todo lo que hicimos hoy desemboca aquí.

### 📘 Para entender — la mesa servida para un modelo

| Lo que el modelo necesita | Lo que hicimos | Función |
|---|---|---|
| Solo números | tonalidad → 12 columnas de 0/1; modo ya era 0/1 | `pd.get_dummies(..., dtype=int)` |
| Sin vacíos | quitar 4 filas sin nombre (hoy); rellenar (miércoles) | `dropna`, `fillna` |
| Sin filas "repetidas" ni imposibles | duplicados por `track_id`; tempo 0 | `drop_duplicates`, filtros |
| Escalas comparables (según el modelo) | z-score ajustado con entrenamiento | `StandardScaler().fit(X_train)` + `transform` |
| Separar antes de aprender | 80 % entrenamiento / 20 % prueba, estratificado | `train_test_split(..., stratify=y)` |

## 5. Epílogo: la máquina escucha números (≈ 5 min)

El mismo árbol de decisión de la Sesión 1, con la línea base al lado. Los árboles no necesitan escalado, pero `X_train_esc` ya está lista y funciona igual.

### 🔍 Predice
La línea base acierta el 19 %. ¿Cuánto acertará el árbol adivinando entre **seis** géneros con solo los números del audio? ¿30 %? ¿50 %? ¿80 %?

**Su predicción:** ____ %

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

tonto = DummyClassifier(strategy="most_frequent").fit(X_train_esc, y_train)
arbol = DecisionTreeClassifier(max_depth=8, random_state=42).fit(X_train_esc, y_train)

print(f"Línea base (siempre '{tonto.predict(X_test_esc)[0]}'): {tonto.score(X_test_esc, y_test):.3f}")
print(f"Árbol de decisión (profundidad 8):     {arbol.score(X_test_esc, y_test):.3f}")

### Una fila entra, una predicción sale
El modelo entrenado es una **función**: le damos una canción (su fila de números) y devuelve un género, con una probabilidad por clase. Probemos con cuatro canciones conocidas. Recuerde que la etiqueta de la tabla la puso **la playlist**: Despacito y *Smells Like Teen Spirit* quedaron como `pop`.

### 🔍 Predice
¿En cuál de las cuatro cree que el árbol va a estar más seguro? ¿En cuál va a dudar?

In [ ]:
X_esc = pd.concat([X_train_esc, X_test_esc])             # todas las filas escaladas, para buscar por índice
pruebas = ["Blinding Lights", "Despacito", "Smells Like Teen Spirit", "Bohemian Rhapsody - Remastered 2011"]
filas = []
for titulo in pruebas:
    i = df.index[df["cancion"] == titulo][0]
    proba = pd.Series(arbol.predict_proba(X_esc.loc[[i]])[0], index=arbol.classes_).sort_values(ascending=False)
    filas.append({"cancion": titulo, "artista": df.loc[i, "artista"], "etiqueta (playlist)": y.loc[i],
                  "predicción": proba.index[0], "probabilidad": round(proba.iloc[0], 2), "segunda opción": proba.index[1]})
pd.DataFrame(filas)

### 🔍 Predice
¿Qué género será el **más fácil** de reconocer para el árbol y cuál el **más difícil**? Piense en cuál suena más distinto a los demás y cuál se parece a varios a la vez.

In [ ]:
aciertos = (arbol.predict(X_test_esc) == y_test)                  # True/False por canción de prueba
aciertos.groupby(y_test).mean().sort_values(ascending=False).round(2)    # proporción de aciertos por género real

Hoy no explicamos el árbol (eso fue la Sesión 1 y vuelve en la 6); el punto es otro: **la tabla que usted preparó hoy es la que hace posible el modelo**. En el curso, cada vez que vea `.fit(X_train, y_train)`, todo lo de las secciones 2 y 4 ya pasó antes.

## 6. Cierre (≈ 7 min)

### Lo que hicimos hoy, con los dos sombreros puestos

| Sombrero | Paso | Hoy | Herramientas |
|---|---|---|---|
| 🔧 | conseguir | CSV desde el repositorio | `pd.read_csv` |
| 🔧 | entender | ¿qué es una fila?, tipos, resumen | `shape`, `head`, `info`, `describe` |
| 🔧 | limpiar | nombres, fechas, duplicados, vacíos, imposibles | `rename`, `astype`, `to_datetime`, `drop_duplicates`, `dropna`, filtros |
| 🔧 | transformar | seleccionar, filtrar, ordenar, contar, agrupar | `df[...]`, `sort_values`, `value_counts`, `groupby` |
| 🔧 | juntar / entregar | tabla de géneros; archivo limpio + bitácora | `merge`, `map`, `to_csv` |
| 🔬 | preguntar y graficar | la receta de 4 pasos; línea, barras, histogramas superpuestos, cajas, dispersión, dos series, `subplots`, guardar; correlación | `plot`, `plot.bar`, `plot.barh`, `plot.hist`, `boxplot`, `plot.scatter`, `plt.subplots`, `plt.savefig`, `corr` |
| 🔬 | preparar X e y | one-hot, estandarizar con entrenamiento, separar | `get_dummies`, `StandardScaler`, `train_test_split` |
| 🔬 | modelar | árbol vs línea base; una fila entra, una predicción sale | `DecisionTreeClassifier`, `DummyClassifier`, `predict_proba` |

### 📘 La chuleta de hoy (cubre el 90 % de lo que verá en las sesiones 2 a 12)

| Quiero… | Python / pandas |
|---|---|
| guardar un valor | `x = 3.5` · `nombre = "Blinding Lights"` |
| ver qué tipo es / convertir | `type(x)` · `int("2019")` · `float("0,8".replace(",", "."))` · `str(3)` |
| texto con variables | `f"{cancion} dura {minutos} min"` |
| una lista y sus posiciones | `[a, b, c]` · `lista[0]` · `lista[-1]` · `lista[1:3]` · `len(lista)` · `lista.append(x)` |
| un registro con nombres | `{"artista": "The Weeknd", "anio": 2019}` · `dic["artista"]` |
| decidir / repetir | `if cond:` … `elif:` … `else:` · `for x in lista:` |
| una función | `def nombre(parametro):` … `return resultado` |
| leer / guardar una tabla | `pd.read_csv("archivo.csv")` · `df.to_csv("salida.csv", index=False)` |
| mirar la tabla | `df.shape` · `df.head()` · `df.info()` · `df.describe()` · `df.dtypes` |
| una o varias columnas | `df["col"]` · `df[["a", "b"]]` |
| filtrar filas | `df[df["col"] > 5]` · `df[(c1) & (c2)]` (para "o", la barra vertical en vez de `&`) · `df[df["col"].str.contains("texto", case=False)]` |
| columna nueva | `df["min"] = df["ms"] / 60000` · `df["cat"] = df["x"].apply(funcion)` · `df["y"] = df["x"].map(diccionario)` |
| ordenar / contar / agrupar | `df.sort_values("col", ascending=False)` · `df["col"].value_counts()` · `df.groupby("g")["x"].mean()` |
| vacíos / duplicados | `df.isna().sum()` · `df.dropna(subset=[...])` · `df.fillna(valor)` · `df.drop_duplicates(subset="id")` |
| fechas | `pd.to_datetime(df["f"], errors="coerce")` · `.dt.year` · `.str[:4]` |
| juntar tablas | `df.merge(otra, on="llave", how="left")` · `pd.concat([a, b], axis=1)` |
| una gráfica (receta) | 1) datos pequeños → 2) `serie.plot()` · `.plot.bar(rot=45)` · `.plot.barh()` · `.plot.hist(bins=30)` · `df.plot.scatter(x=, y=, alpha=)` · `df.boxplot(column=, by=)` → 3) `plt.title()` · `plt.xlabel()` · `plt.ylabel()` → 4) `figsize=`, `color=`, `plt.grid(alpha=0.3)`, `plt.show()` |
| varias series / varias gráficas | dos llamadas seguidas con `label=` + `plt.legend()` · `fig, axes = plt.subplots(1, 2, figsize=(12, 4))` y `ax=axes[0]` · `plt.tight_layout()` |
| guardar una gráfica | `plt.savefig("nombre.png", dpi=150, bbox_inches="tight")` antes de `plt.show()` |
| preparar para un modelo | `pd.get_dummies(df["cat"], dtype=int)` · `StandardScaler().fit(X_train)` · `.transform(X)` · `train_test_split(X, y, test_size=0.2, stratify=y)` |

### 📘 Cómo leer un error (y qué hacer)

| Último renglón del error | Qué significa casi siempre | Qué hacer |
|---|---|---|
| `NameError: name 'x' is not defined` | la variable no existe (no corrió la celda que la crea, o está mal escrita) | correr las celdas anteriores; revisar mayúsculas |
| `SyntaxError` / `IndentationError` | falta un paréntesis, una comilla, dos puntos; o la sangría está mal | mirar la flecha `^`; contar paréntesis |
| `TypeError` | mezcló tipos (texto + número) o llamó algo con argumentos equivocados | `str()`, `int()`, `float()`; leer el mensaje completo |
| `KeyError: 'col'` | la columna (o llave) no existe con ese nombre | `df.columns`; revisar tildes, espacios, mayúsculas |
| `IndexError` | pidió una posición que la lista no tiene | recordar que se cuenta desde 0 |
| `ValueError` | el valor no sirve para esa operación (`float("0,8")`, `NaN` en un modelo) | convertir, limpiar vacíos |
| `AttributeError: ... has no attribute` | ese método no existe para ese objeto (p. ej. `.mean()` en un texto) | revisar el tipo; `df["col"].str.` para métodos de texto |
| `FileNotFoundError` / `ModuleNotFoundError` | el archivo no está donde se busca / la librería no está instalada | volver a correr el Setup; `!pip install` |

**Método:** lea el último renglón, mire en qué línea de *su* celda ocurrió, y si no lo resuelve en dos minutos, copie ese último renglón en un buscador o en un asistente de IA. Lo hacen todos los profesionales, todos los días. Lo que no se hace es ignorarlo.

### Para esta semana (antes del miércoles)

1. **Vuelva a correr este notebook solo**, de arriba abajo, y haga las 🧪. Lo que no entienda, anótelo y tráigalo.
1b. **Haga dos gráficas propias** con la receta de cuatro pasos (sección 3): una pregunta suya, una columna distinta, un tipo distinto. Guárdelas con `plt.savefig`.
2. **Haga (o repita) el Notebook 0** del curso: ahora debería salirle entero.
3. **Relea el notebook de la Sesión 1** y marque la primera línea que no entienda. El miércoles empezamos por ahí.
4. Si quiere practicar más, en este orden:
   - [Tutorial oficial de Python en español](https://docs.python.org/es/3/tutorial/) — capítulos 3 a 5 (una hora).
   - [Kaggle Learn: Python](https://www.kaggle.com/learn/python) y [Kaggle Learn: Pandas](https://www.kaggle.com/learn/pandas) — gratis, en inglés, con ejercicios que se corrigen solos en el navegador (4–5 horas cada uno).
   - [10 minutos con pandas](https://pandas.pydata.org/docs/user_guide/10min.html) — la guía oficial, para tener a la mano.

**El miércoles (Sesión 2):** volvemos a los incidentes viales de Medellín. Verá tipos de variable, vacíos disfrazados de texto, distribuciones, *one-hot* y escalado otra vez (ahora les va a sonar), y un árbol que aprendió algo que no debía. Todo lo de hoy reaparece.

### ⚡ Reto final (opcional): otro dataset, los dos sombreros

Dos tablas sobre los **volcanes del mundo** (Smithsonian Global Volcanism Program, vía TidyTuesday 2020-05-12): `volcanes.csv` (958 volcanes: nombre, país, tipo, altura, coordenadas, población alrededor) y `erupciones.csv` (11 178 erupciones: volcán, año de inicio, índice de explosividad `vei`). Ya están descargadas por el Setup.

1. 🔧 Cargue las dos tablas; `shape`, `head`, `info`. ¿Qué representa una fila en cada una?
2. 🔧 ¿Cuántos volcanes tiene **Colombia** y cómo se llaman? (filtrar por `country`).
3. 🔧 Una `merge` entre las dos por `volcano_number` y cuente erupciones por volcán colombiano. ¿Cuál es el más activo?
4. 🔬 Dibuje `df.plot.scatter(x="longitude", y="latitude", s=3)` con los 958 volcanes. ¿Qué forma aparece?
5. 🔬 ¿Cuántas personas viven a menos de 30 km del Galeras (`population_within_30_km`)? ¿Y qué `vei` tuvo la erupción del Nevado del Ruiz de 1985?

<details><summary>Pistas</summary>

- `volcanes = pd.read_csv("volcanes.csv")`, `erupciones = pd.read_csv("erupciones.csv")`.
- `volcanes[volcanes["country"] == "Colombia"][["volcano_name", "primary_volcano_type", "elevation", "last_eruption_year"]]`.
- `juntas = erupciones.merge(volcanes[["volcano_number", "country"]], on="volcano_number")` y luego `juntas[juntas["country"] == "Colombia"]["volcano_name"].value_counts()`.
- Para 1985: filtre `juntas` por `volcano_name` que contenga "Ruiz" y `start_year == 1985`.
- El dibujo del punto 4 es el **Cinturón de Fuego del Pacífico**, con dos líneas de código.
</details>

In [ ]:
# Espacio para el reto final


### 📘 La sesión en una página (para repasar)

| Concepto | Qué es | En código |
|---|---|---|
| Variable y tipo | un valor con nombre; el tipo decide qué se puede hacer | `x = 3.5` · `type(x)` |
| Lista | valores en orden, posiciones desde 0 | `lista[0]`, `lista[-1]`, `lista[1:3]` |
| Diccionario | valores con nombre (llave → valor) | `dic["llave"]` |
| DataFrame | tabla: una lista de diccionarios con las mismas llaves | `pd.DataFrame(lista)`, `pd.read_csv(...)` |
| `if` / `for` / `def` | decidir, repetir, empaquetar | sangría de 4 espacios; `return` |
| Los cinco pasos de limpieza | nombres, tipos y fechas, duplicados, vacíos, imposibles | `rename`, `astype`, `drop_duplicates`, `dropna`, filtros |
| ¿Qué representa una fila? | la pregunta que evita contar *Blinding Lights* ocho veces | `duplicated()`, `drop_duplicates(subset=...)` |
| Las cinco operaciones | seleccionar, filtrar, ordenar, contar, agrupar | `df[...]`, `sort_values`, `value_counts`, `groupby` |
| `merge` | buscar en otra tabla por una llave | `df.merge(otra, on="llave", how="left")` |
| Una gráfica por pregunta | receta: datos → tipo → títulos → ajustes; línea, barras ordenadas, histograma/cajas, dispersión; barras siempre desde cero | `.plot`, `.plot.barh`, `.plot.hist`, `boxplot`, `.plot.scatter`, `plt.subplots`, `plt.savefig` |
| Correlación | de −1 a 1; no es causa | `df[cols].corr()` |
| One-hot | categoría → columnas de 0/1 (también las disfrazadas de número) | `pd.get_dummies(..., dtype=int)` |
| Estandarización | $z = (x - \text{media}) / \text{desviación}$, ajustada con entrenamiento | `StandardScaler().fit(X_train)` |
| Línea base | lo que acierta un modelo tonto; si no se le gana, no hay modelo | `DummyClassifier` |
